In [512]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import chart_studio
import chart_studio.plotly as py
import plotly.io as pio
import re, string, unicodedata
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', -1)


In [513]:
username = 'politicai' # your username
api_key = 'REDACTED' # your api key - go to profile > settings > regenerate key
chart_studio.tools.set_credentials_file(username=username, api_key=api_key)
# py.plot(fig_object, filename = 'gdp_per_cap', auto_open=True)
# pio.write_html(fig, file=’index.html’, auto_open=True)

In [514]:
def clean_df_format(df):
    df.columns = [x.lower().replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "").strip() for x in df.columns]
    return df

def clean_country_names(df, column, new_country_col_name="country"):
    df[column] = df[column].str.lower()
    df[column] = df[column].str.replace("(", "").str.replace(")", "").str.replace("&", "")
    df[column] = df[column].str.replace(".", "").str.replace(",", "").str.replace("'","")
    df[column] = df[column].str.replace("-", " ").str.replace(" ", "_").str.replace("__", "_")
    
    name_changes={"korea_north":"north_korea", "korea_dem_people’s_rep":"north_korea", "venezuela_rb":"venezuela",
                  "korea_south":"south_korea","korea_rep":"south_korea" ,"cape_verde":"cabo_verde", "syrian_arab_republic":"syria",
                 "macedonia":"north_macedonia", "serbia_montenegro":"serbia", "slovak_republic":"slovakia",
                 "dem_rep_of_congo":"congo_kinshasa", "congo_dem_rep":"congo_kinshasa","congo_rep":"congo_brazzaville",
                  "egypt_arab_rep":"egypt", "gambia_the":"gambia","the_gambia":"gambia" ,"russian_federation":"russia",
                 "iran_islamic_rep":"iran", "yemen_rep":"yemen", "kyrgyz_republic":"kyrgyzstan", 
                 "lao_pdr":"laos", "bahamas_the":"bahamas", "brunei_darussalam":"brunei", "sudan_north":"sudan", 
                 "hong_kong_sar_china":"hong_kong"}
    df[column] =  df[column].replace(name_changes)

    df = df.rename(columns={column:new_country_col_name})
    return df

# Cargar ficheros

In [515]:
df_fragility = pd.read_excel("../../data/staging/mundial/fragilidad.xls")
df_fragility = clean_df_format(df_fragility)

In [516]:
df_freedomhouse = pd.read_excel("../../data/staging/mundial/freedom_house.xlsx")
df_freedomhouse = clean_df_format(df_freedomhouse)

In [517]:
df_poblacionactiva = pd.read_excel("../../data/staging/mundial/tasa_poblacion_activa.xls")
df_poblacionactiva = clean_df_format(df_poblacionactiva)

In [518]:
df_esperanzavida = pd.read_excel("../../data/staging/mundial/esperanza_vida.xls")
df_esperanzavida = clean_df_format(df_esperanzavida)

In [519]:
df_escolarizacion = pd.read_csv("../../data/staging/mundial/años_escolarizacion_medios.csv", sep=",")
df_escolarizacion = clean_df_format(df_escolarizacion)

In [520]:
df_sistemaselectorales = pd.read_excel("../../data/staging/mundial/sistemas_electorales.xls")
df_sistemaselectorales = clean_df_format(df_sistemaselectorales)

In [521]:
df_poblacion = pd.read_excel("../../data/staging/mundial/poblacion.xls")
df_poblacion = clean_df_format(df_poblacion)

In [522]:
# COUNTRY,YEAR,HOSELECT,HOSTERM,HOSDISS,HOGELECT,HOGTERM,HOGDISS,LEGISL,HOUSENUM,FEDUNIT
#df_constituciones = pd.read_excel("../../data/staging/mundial/constituciones.xlsx")
#df_constituciones = df_constituciones[["country","year","hoselect","hosterm","hosdiss","hogelect","hogterm","hogdiss","legisl","housenum","fedunit"]]
#df_constituciones.to_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",", index=False)
df_constituciones_red = pd.read_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",")
df_constituciones_red = clean_df_format(df_constituciones_red)

# Tabla Macro

## Fragility

In [523]:
df_base = df_fragility[["scode", "country", "year", "sfi"]]
df_base = df_base.rename(columns={"scode":"code", "sfi":"state_fragility_idx"})
df_base = clean_country_names(df=df_base, column="country")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  167
filas:  (3960, 4)


,code,country,year,state_fragility_idx
0,AFG,afghanistan,1995,25
1,AFG,afghanistan,1996,24
2,AFG,afghanistan,1997,24
3,AFG,afghanistan,1998,24
4,AFG,afghanistan,1999,24


### Freedom in the world

In [524]:
df_freedomhouse_idx = df_freedomhouse[["country_territory", "status", "total", "edition"]]
df_freedomhouse_idx = df_freedomhouse_idx.rename(columns={"status":"free_not_free", "total":"democracy_idx",
                                                         "edition":"year"})
df_freedomhouse_idx = clean_country_names(df=df_freedomhouse_idx, column="country_territory")
df_freedomhouse_idx.head()

,country,free_not_free,democracy_idx,year
0,abkhazia,PF,40,2020
1,afghanistan,NF,27,2020
2,albania,PF,67,2020
3,algeria,NF,34,2020
4,andorra,F,94,2020


In [525]:
df_base = df_base.merge(right=df_freedomhouse_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  166
filas:  (996, 6)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx
0,AFG,afghanistan,2013,21,NF,27
1,AFG,afghanistan,2014,21,NF,26
2,AFG,afghanistan,2015,21,NF,24
3,AFG,afghanistan,2016,21,NF,24
4,AFG,afghanistan,2017,21,NF,24


### Poblacion activa

In [526]:
df_poblacionactiva_idx = pd.melt(df_poblacionactiva, id_vars=['country_name','country_code'], value_vars=df_poblacionactiva.columns[4:])
df_poblacionactiva_idx = df_poblacionactiva_idx[["country_name", "country_code", "variable", "value"]]
df_poblacionactiva_idx = df_poblacionactiva_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"tasa_pob_activa"})
df_poblacionactiva_idx = clean_country_names(df=df_poblacionactiva_idx, column="country_name")
df_poblacionactiva_idx.year = df_poblacionactiva_idx.year.astype(int)

df_poblacionactiva_idx.head()

,country,code_wb,year,tasa_pob_activa
0,aruba,ABW,1960,NaN
1,afghanistan,AFG,1960,NaN
2,angola,AGO,1960,NaN
3,albania,ALB,1960,NaN
4,andorra,AND,1960,NaN


In [527]:
df_base = df_base.merge(right=df_poblacionactiva_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  165
filas:  (990, 8)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000


### Esperanza de vida

In [528]:
df_esperanzavida_idx = pd.melt(df_esperanzavida, id_vars=['country_name','country_code'], value_vars=df_esperanzavida.columns[4:])
df_esperanzavida_idx = df_esperanzavida_idx[["country_code", "variable", "value"]]
df_esperanzavida_idx = df_esperanzavida_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"esperanza_vida"})
df_esperanzavida_idx.year = df_esperanzavida_idx.year.astype(int)

df_esperanzavida_idx.head()

,code_wb,year,esperanza_vida
0,ABW,1960,65.662
1,AFG,1960,32.446
2,AGO,1960,37.524
3,ALB,1960,62.283
4,AND,1960,NaN


In [529]:
df_base = df_base.merge(right=df_esperanzavida_idx, on=["code_wb", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  165
filas:  (990, 9)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130


### Años escolarización

In [530]:
df_escolarizacion_idx = df_escolarizacion
df_escolarizacion_idx = df_escolarizacion_idx.rename(columns={"code":"code_un", 
                                                              "average_total_years_of_schooling_for_adult_population":"anios_escolarizacion",})

df_escolarizacion_idx = clean_country_names(df=df_escolarizacion_idx, column="entity")
df_escolarizacion_idx.year = df_escolarizacion_idx.year.astype(int)
df_escolarizacion_idx.head()

,country,code_un,year,anios_escolarizacion
0,afghanistan,AFG,1870,0.01
1,afghanistan,AFG,1875,0.01
2,afghanistan,AFG,1880,0.01
3,afghanistan,AFG,1885,0.01
4,afghanistan,AFG,1890,0.01


In [531]:
df_base = df_base.merge(right=df_escolarizacion_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  159
filas:  (795, 11)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525,AFG,3.5
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966,AFG,3.5
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377,AFG,3.6
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763,AFG,3.6
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130,AFG,3.8


### Sistema electoral

In [532]:
df_sistemaselectorales_idx = df_sistemaselectorales
df_sistemaselectorales_idx = clean_country_names(df=df_sistemaselectorales_idx, column="country")
df_sistemaselectorales_idx.year = df_sistemaselectorales_idx.year.astype(int)
df_sistemaselectorales_idx.head()

,country,year,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president
0,afghanistan,2018,Other,SNTV,1,250.0,250.0,TRS
1,afghanistan,2010,Other,SNTV,1,249.0,249.0,TRS
2,afghanistan,2005,Other,SNTV,1,249.0,249.0,TRS
3,albania,2017,PR,List PR,1,140.0,140.0,Not applicable
4,albania,2013,PR,List PR,1,140.0,140.0,Not applicable


In [533]:
df_base = df_base.merge(right=df_sistemaselectorales_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  144
filas:  (170, 17)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president
0,ALB,albania,2013,3,PF,63,ALB,52.308998,77.554,ALB,9.7,PR,List PR,1,140.0,140.0,Not applicable
1,ALB,albania,2017,1,PF,68,ALB,55.908001,78.333,ALB,10.0,PR,List PR,1,140.0,140.0,Not applicable
2,ALG,algeria,2017,11,NF,35,DZA,41.383999,76.499,DZA,8.0,PR,List PR,1,462.0,462.0,TRS
3,ANG,angola,2017,16,NF,24,AGO,77.568001,60.379,AGO,5.1,PR,List PR,2,220.0,220.0,Other
4,ARG,argentina,2013,1,F,80,ARG,60.220001,75.756,ARG,9.8,PR,List PR,1,257.0,257.0,TRS


### Población

In [535]:
df_poblacion_idx = pd.melt(df_poblacion, id_vars=['country_name','country_code'], value_vars=df_poblacion.columns[4:])
df_poblacion_idx = df_poblacion_idx[["country_code", "variable", "value"]]
df_poblacion_idx = df_poblacion_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"poblacion"})
df_poblacion_idx.year = df_poblacion_idx.year.astype(int)
df_poblacion_idx.head()

,code_wb,year,poblacion
0,ABW,1960,54211.0
1,AFG,1960,8996973.0
2,AGO,1960,5454933.0
3,ALB,1960,1608800.0
4,AND,1960,13411.0


In [537]:
df_base = df_base.merge(right=df_poblacion_idx, on=["code_wb", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  144
filas:  (170, 18)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president,poblacion
0,ALB,albania,2013,3,PF,63,ALB,52.308998,77.554,ALB,9.7,PR,List PR,1,140.0,140.0,Not applicable,2895092.0
1,ALB,albania,2017,1,PF,68,ALB,55.908001,78.333,ALB,10.0,PR,List PR,1,140.0,140.0,Not applicable,2873457.0
2,ALG,algeria,2017,11,NF,35,DZA,41.383999,76.499,DZA,8.0,PR,List PR,1,462.0,462.0,TRS,41389198.0
3,ANG,angola,2017,16,NF,24,AGO,77.568001,60.379,AGO,5.1,PR,List PR,2,220.0,220.0,Other,29816748.0
4,ARG,argentina,2013,1,F,80,ARG,60.220001,75.756,ARG,9.8,PR,List PR,1,257.0,257.0,TRS,42202935.0


### Constituciones

In [540]:
df_constituciones_red_idx = df_constituciones_red
df_constituciones_red_idx = clean_country_names(df=df_constituciones_red_idx, column="country")
df_constituciones_red_idx.year = df_constituciones_red_idx.year.astype(int)
df_constituciones_red_idx.head()

,country,year,hoselect,hosterm,hosdiss,hogelect,hogterm,hogdiss,legisl,housenum,fedunit
0,abkhazia,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abkhazia,2009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,abkhazia,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,abkhazia,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,abkhazia,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [541]:
df_base = df_base.merge(right=df_constituciones_red_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  30
filas:  (30, 27)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president,poblacion,hoselect,hosterm,hosdiss,hogelect,hogterm,hogdiss,legisl,housenum,fedunit
0,ALB,albania,2013,3,PF,63,ALB,52.308998,77.554000,ALB,9.7,PR,List PR,1,140.0,140.0,Not applicable,2895092.0,3.0,5,1.0,4.0,0,1.0,1.0,2.0,NaN
1,ARG,argentina,2013,1,F,80,ARG,60.220001,75.756000,ARG,9.8,PR,List PR,1,257.0,257.0,TRS,42202935.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AUL,australia,2013,2,F,97,AUS,65.095001,82.148780,AUS,12.6,Plurality/Majority,AV,1,150.0,150.0,Not applicable,23128129.0,1.0,0,2.0,99.0,99,99.0,1.0,3.0,1.0
3,AUS,austria,2013,0,F,96,AUT,60.432999,81.136585,AUT,11.9,PR,List PR,3,183.0,183.0,TRS,8479823.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BHU,bhutan,2013,8,PF,46,BTN,64.269997,69.662000,BTN,2.8,Plurality/Majority,FPTP,1,47.0,47.0,Not applicable,710238.0,1.0,100,1.0,3.0,5,1.0,1.0,3.0,NaN


In [544]:
df_constituciones_red_idx.country.unique()

array(['abkhazia', 'afghanistan', 'albania', 'algeria', 'andorra',
       'angola', 'antigua_and_barbuda', 'argentina', 'armenia',
       'australia', 'austria', 'austria_hungary', 'azerbaijan', 'baden',
       'bahamas', 'bahrain', 'bangladesh', 'barbados', 'bavaria',
       'belarus_byelorussia', 'belgium', 'belize', 'benin', 'bhutan',
       'bolivia', 'bosnia_herzegovina', 'botswana', 'brazil', 'brunei',
       'bulgaria', 'burkina_faso_upper_volta', 'burundi',
       'cambodia_kampuchea', 'cameroon', 'canada', 'cabo_verde',
       'central_african_republic', 'chad', 'chile', 'china', 'colombia',
       'comoros', 'congo', 'congo_democratic_republic_of_zaire',
       'costa_rica', 'cote_divoire', 'croatia', 'cuba', 'cyprus',
       'czech_republic', 'czechoslovakia', 'denmark', 'djibouti',
       'dominica', 'dominican_republic', 'east_timor', 'ecuador', 'egypt',
       'el_salvador', 'equatorial_guinea', 'eritrea', 'estonia',
       'ethiopia', 'fiji', 'finland', 'france', 'gabon'